# Movie Recommendation — User-User Collaborative Filtering

**Objective:** recommend a movie to a user based on what their most similar user rated highly.
**Method:** cosine similarity over a user x movie rating matrix.

**Data:** a small hand-built rating table (6 users) to keep the mechanism easy to trace end to end — see the project README for the planned upgrade to a real dataset.

In [1]:
import pandas as pd

raw_data = [
    ['Alice', 'Inception', 5], ['Alice', 'Interstellar', 4], ['Alice', 'Titanic', 1],
    ['Bob', 'Inception', 4], ['Bob', 'Interstellar', 5], ['Bob', 'The Matrix', 2], ['Bob', 'Titanic', 5], ['Bob', 'Avatar', 5],
    ['Charlie', 'The Godfather', 5], ['Charlie', 'Pulp Fiction', 4], ['Charlie', 'Inception', 1],
    ['David', 'The Godfather', 2], ['David', 'Pulp Fiction', 5], ['David', 'Titanic', 4],
    ['Eve', 'The Godfather', 5], ['Eve', 'Pulp Fiction', 4], ['Eve', 'Interstellar', 2],
    ['Frank', 'The Matrix', 5], ['Frank', 'Inception', 4], ['Frank', 'Interstellar', 4]
]

df = pd.DataFrame(raw_data, columns=['User', 'Movie', 'Rating'])
df

,User,Movie,Rating
0,Alice,Inception,5
1,Alice,Interstellar,4
2,Alice,Titanic,1
3,Bob,Inception,4
4,Bob,Interstellar,5
5,Bob,The Matrix,2
6,Bob,Titanic,5
7,Bob,Avatar,5
8,Charlie,The Godfather,5
9,Charlie,Pulp Fiction,4


## STEP 01 - Build the User x Movie Matrix

Only ratings of 4+ are kept, so the matrix reflects what a user liked, not everything they rated.

In [2]:
df_high_rated = df[df['Rating'] >= 4].copy()
matrix = df_high_rated.pivot(index='User', columns='Movie', values='Rating').fillna(0)
matrix

Movie,Avatar,Inception,Interstellar,Pulp Fiction,The Godfather,The Matrix,Titanic
User,,,,,,,
Alice,0.0,5.0,4.0,0.0,0.0,0.0,0.0
Bob,5.0,4.0,5.0,0.0,0.0,0.0,5.0
Charlie,0.0,0.0,0.0,4.0,5.0,0.0,0.0
David,0.0,0.0,0.0,5.0,0.0,0.0,4.0
Eve,0.0,0.0,0.0,4.0,5.0,0.0,0.0
Frank,0.0,4.0,4.0,0.0,0.0,5.0,0.0


## STEP 02 - User-User Cosine Similarity

Cosine similarity over correlation: it looks at rating *pattern* rather than absolute scale, and handles the sparse "0 = unwatched" structure without those zeros being treated as a real low rating.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity

user_sim = cosine_similarity(matrix)
user_sim_df = pd.DataFrame(user_sim, index=matrix.index, columns=matrix.index)
user_sim_df

User,Alice,Bob,Charlie,David,Eve,Frank
User,,,,,,
Alice,1.000000,0.654858,0.000000,0.000000,0.000000,0.744686
Bob,0.654858,1.000000,0.000000,0.327429,0.000000,0.499855
Charlie,0.000000,0.000000,1.000000,0.487805,1.000000,0.000000
David,0.000000,0.327429,0.487805,1.000000,0.487805,0.000000
Eve,0.000000,0.000000,1.000000,0.487805,1.000000,0.000000
Frank,0.744686,0.499855,0.000000,0.000000,0.000000,1.000000


## STEP 03 - Recommend from the Most Similar User

In [4]:
def recommend_smart(target_user):
    sim_series = user_sim_df[target_user].sort_values(ascending=False)
    similar_user = sim_series.index[1]
    similarity_score = sim_series.iloc[1]

    target_seen = df[df['User'] == target_user]['Movie'].unique()
    sim_user_loves = df[(df['User'] == similar_user) & (df['Rating'] >= 4)]
    recommendations = sim_user_loves[~sim_user_loves['Movie'].isin(target_seen)]

    print(f"Target User: {target_user}")
    print(f"Matched with: {similar_user} (Similarity Score: {similarity_score:.2f})")

    if not recommendations.empty:
        print(f"Recommended Movies: {recommendations['Movie'].tolist()}")
    else:
        print("Recommended Movies: No new high-rated movies found from this match.")

recommend_smart('Alice')

Target User: Alice
Matched with: Frank (Similarity Score: 0.74)
Recommended Movies: ['The Matrix']
